In [0]:
def bc_ev_harmonized():
    from pyspark.sql import functions as F
    df_bc_silver = (spark.readStream
                    .table("ev_spark.silver.bc_ev")
    )

    df_postal_codes = spark.table("ev_spark.silver.postal_codes")

    df_bc_harmonized = (df_bc_silver
                            .join(
                                df_postal_codes,
                                F.lower(df_bc_silver.city) == F.lower(df_postal_codes.City),
                                "inner"
                            )
                            .select(df_bc_silver.province, "fsa", "ev_count")
    )
    return df_bc_harmonized

In [0]:
def fsaCenter():
    from pyspark.sql import functions as F
    geoCodes_df = spark.table("ev_spark.silver.postal_codes")
    center_df = (geoCodes_df
                .withColumn("fsa", F.substring(F.col("fsa"),1,3))
                .groupBy("fsa")
                .agg(
                F.avg("Latitude").alias("center_lat"),
                F.avg("Longitude").alias("center_lon")
                )
    )
    postalCodes_df = (spark.table("ev_spark.silver.postal_codes")
                    .withColumn("fsa", F.substring(F.col("fsa"),1,3))
    )

    postalCodesjoin_df = (postalCodes_df
                    .join(center_df, on="fsa", how="inner")
    )
    return postalCodesjoin_df


In [0]:
def fsaCity(df):
    from pyspark.sql import functions as F
    from pyspark.sql.window import Window
    import math


    R = 6371.0

    df_with_dist = (df
                    .withColumn("distance_km",
                                2 * R * F.asin(
                                    F.sqrt(
                                        F.pow(F.sin((F.radians(F.col("Latitude")) - F.radians(F.col("center_lat"))) / 2), 2) +
                                        F.cos(F.radians(F.col("center_lat"))) *
                                        F.cos(F.radians(F.col("Latitude"))) *
                                        F.pow(F.sin((F.radians(F.col("Longitude")) - F.radians(F.col("center_lon"))) / 2), 2)
                                    )
                                )
                                )
                    )

    window_spec = Window.partitionBy("fsa").orderBy(F.col("distance_km").asc())
    df_with_dist = df_with_dist.withColumn("rank", F.row_number().over(window_spec))
    df_with_dist = df_with_dist.filter(F.col("rank") == 1)
    return df_with_dist

In [0]:
def on_ev_harmonized(df):
    from pyspark.sql import functions as F
    df_on_silver = (spark.readStream
                    .table("ev_spark.silver.ontario_ev")
    )

    df_on_harmonized = (df_on_silver
                        .join(df, on=["fsa"], how="inner")
                        .select(df_on_silver.province, "fsa", "ev_count")
    )
    return df_on_harmonized


In [0]:
def evCombined():
    df_bc_harmonized = bc_ev_harmonized()
    postalCodesjoin_df = fsaCenter()
    df_fsaCity = fsaCity(postalCodesjoin_df)
    df_on_harmonized = on_ev_harmonized(df_fsaCity)
    df_ev_combined = (df_bc_harmonized
                        .unionByName(df_on_harmonized)
    )
    return df_ev_combined
    

In [0]:
def writeEvCombinedToSilver(df):
    (df
        .writeStream
        .format("delta")
        .option("checkpointLocation", "/Volumes/ev_spark/myvol/checkpoint/chkpt/ev_combined_silver/")
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable("ev_spark.silver.ev_combined")
    )

In [0]:
df_bc_harmonized = bc_ev_harmonized()
df_fsaCenter = fsaCenter()
df_fsaCity = fsaCity(df_fsaCenter)
on_ev_harmonized(df_fsaCity)
df_ev_combined = evCombined()
writeEvCombinedToSilver(df_ev_combined)